In [31]:
# Librerías básicas
import re
import os
import glob
import unicodedata
import docx2txt


In [32]:
# STEP 1: Load documents
# Directory containing the .docx files
DATA_DIR = "../data"

In [33]:
!pip install python-docx



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
# Install docx2txt if not already installed
!pip install docx2txt

# Find all .docx files in the data directory
docx_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.docx")))
print(f"Found {len(docx_files)} .docx files:")
for f in docx_files:
    print(f"  {os.path.basename(f)}")

# Read and concatenate all documents
all_texts = []
for filepath in docx_files:
    doc_text = docx2txt.process(filepath)
    all_texts.append(doc_text)
    print(f"Read: {os.path.basename(filepath)} ({len(doc_text)} chars)")

# Combine all texts into a single string
text = "\n\n".join(all_texts)

print(f"\nTotal length: {len(text)} chars across {len(docx_files)} files")

Found 13 .docx files:
  243591-DO-AVO-11-V07-251210.docx
  254275-DO-AVO-14-V07.docx
  254275-DO-AVO-15-V07.docx
  254275-DO-AVO-16-V07.docx
  254275-DO-AVO-17-V07.docx
  254275-DO-AVO-18-V07.docx
  254275-DO-AVO-19-V07.docx
  254275-DO-AVO-20-V07.docx
  254275-DO-AVO-21-V01.docx
  254275-DO-AVO-22-V01.docx
  254275-DO-AVO-23-V01.docx
  254275-DO-AVO-24-V01.docx
  254275-DO-AVO-28-V01.docx
Read: 243591-DO-AVO-11-V07-251210.docx (25644 chars)
Read: 254275-DO-AVO-14-V07.docx (11558 chars)
Read: 254275-DO-AVO-15-V07.docx (12401 chars)
Read: 254275-DO-AVO-16-V07.docx (14114 chars)
Read: 254275-DO-AVO-17-V07.docx (17033 chars)
Read: 254275-DO-AVO-18-V07.docx (15480 chars)
Read: 254275-DO-AVO-19-V07.docx (15671 chars)
Read: 254275-DO-AVO-20-V07.docx (13911 chars)
Read: 254275-DO-AVO-21-V01.docx (14660 chars)
Read: 254275-DO-AVO-22-V01.docx (24305 chars)
Read: 254275-DO-AVO-23-V01.docx (19674 chars)
Read: 254275-DO-AVO-24-V01.docx (15766 chars)
Read: 254275-DO-AVO-28-V01.docx (14068 chars)

T


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
# STEP 2: Clean text
def clean_text(text):
    # 1. Unicode normalization: resolves ligatures, full-width chars, composite characters
    text = unicodedata.normalize("NFKC", text)

    # 2. Remove control characters (null bytes, form feeds, etc.) but keep \n and \t
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)

    # 3. Normalize dash variants (em dash, en dash, soft hyphen, minus) to hyphen
    text = re.sub(r'[­‐-―−－]', '-', text)

    # 4. Normalize typographic quotes to straight ASCII quotes
    text = re.sub(r'[‘’‚‛]', "'", text)
    text = re.sub(r'[“”„‟]', '"', text)

    # 5. Replace bullet-point characters with a dash
    text = re.sub(r'[•‣●◦⁃∙·]', '-', text)

    # 6. Remove standalone page numbers (lines containing only digits)
    text = re.sub(r'(?m)^\s*\d{1,4}\s*$', '', text)

    # 7. Remove repeated punctuation runs (e.g. "---", "___", "...")
    text = re.sub(r'([.\-_=])\1{2,}', r'\1', text)

    # 8. Fix spacing before punctuation (e.g. "word ." → "word.")
    text = re.sub(r'\s+([.,;:!?)])', r'\1', text)

    # 9. Collapse all whitespace (spaces, tabs, newlines) to a single space
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [ ]:
cleaned_document_text = clean_text(text)
print("Longitud del texto limpio:", len(cleaned_document_text))
print(cleaned_document_text[0:500])  # Print the first 500 characters of the cleaned text

Longitud del texto limpio: 207246
ACTA DE VISITA DE OBRA No 010 243591-DO-AVO-011-V07_251210 MEJORA DE LA ACCESIBILIDAD DE LA ESTACIÓN DE SANTA PERPÈTUA DE LA MOGODA - LA FLORIDA, BARCELONA Fecha Lugar Hora de inicio Hora de Finalización 10/12/25 Estación de Santa Perpètua de la Mogoda - La Florida. 10:00 13:00 Principales asistentes: Nombre Empresa Cargo E-mail Teléfono Firma opcional FUNCIÓN: DC (director de contrato) DF (dirección Facultativa) DO (dirección de obra) DEO (dirección de ejecución) UTE (contratista) CSS (coordina


In [37]:
# STEP 3: Chunking
def chunk_text(text, chunk_size=500, overlap=100, min_chunk_size=50):
    """
    Split text into overlapping chunks, snapping boundaries to sentences then words.

    Args:
        chunk_size:     target maximum characters per chunk
        overlap:        characters of context shared between consecutive chunks
        min_chunk_size: chunks shorter than this are discarded (avoids tiny tail chunks)
    """
    chunks = []
    if not text:
        return chunks

    stride = chunk_size - overlap
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)

        if end < text_len:
            # 1. Try to end at a sentence boundary in the last 20 % of the window
            search_from = start + int(chunk_size * 0.8)
            best = -1
            for punct in '.!?;':
                pos = text.rfind(punct, search_from, end)
                if pos > best:
                    best = pos
            if best != -1:
                end = best + 1          # include the closing punctuation
            else:
                # 2. Fall back to the nearest word boundary
                space = text.rfind(' ', search_from, end)
                if space != -1:
                    end = space         # cut just before the space

        chunk = text[start:end].strip()
        if len(chunk) >= min_chunk_size:
            chunks.append(chunk)

        start += stride

    return chunks

In [38]:
chunks = chunk_text(cleaned_document_text)
print(f"Número total de chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

Número total de chunks: 518

--- Chunk 1 ---
ACTA DE VISITA DE OBRA No 010 243591-DO-AVO-011-V07_251210 MEJORA DE LA ACCESIBILIDAD DE LA ESTACIÓN DE SANTA PERPÈTUA DE LA MOGODA - LA FLORIDA, BARCELONA Fecha Lugar Hora de inicio Hora de Finalización 10/12/25 Estación de Santa Perpètua de la Mogoda - La Florida. 10:00 13:00 Principales asistentes: Nombre Empresa Cargo E-mail Teléfono Firma opcional FUNCIÓN: DC (director de contrato) DF (dirección Facultativa) DO (dirección de obra) DEO (dirección de ejecución) UTE (contratista) CSS

--- Chunk 2 ---
ión Facultativa) DO (dirección de obra) DEO (dirección de ejecución) UTE (contratista) CSS (coordinador de seguridad) OBJETO Visita Semanal de obra DOCUMENTACIÓN ENTREGADA O A ENTREGAR DOCUMENTACIÓN DE CONSULTA Proyecto Constructivo de Mejora de la accesibilidad de la estación de Santa Perpetua de la Mogoda - La Florida, Barcelona. ASUNTOS TRATADOS, ACUERDOS ALCANZADOS, COMUNICACIONES, INSTRUCCIONES: No Fecha DESCRIPCIÓN DEL ASUNTO: 1. AVANCE 

In [33]:
# Dataset estructurado
data = []

for i, chunk in enumerate(chunks):
    data.append({
        "doc_id": "acta_obra_1",
        "chunk_id": i,
        "text": chunk
    })



In [34]:
print(len(data))
# 16 chunks

55


In [35]:
# Para ver el Data
for item in data:
    print(item)
    print("\n---\n")

{'doc_id': 'acta_obra_1', 'chunk_id': 0, 'text': 'ACTA DE VISITA DE OBRA Nº 010 243591-DO-AVO-011-V07_251210 MEJORA DE LA ACCESIBILIDAD DE LA ESTACIÓN DE SANTA PERPÈTUA DE LA MOGODA – LA FLORIDA, BARCELONA Fecha Lugar Hora de inicio Hora de Finalización 10/12/25 Estación de Santa Perpètua de la Mogoda - La Florida. 10:00 13:00 Principales asistentes: Nombre Empresa Cargo E-mail Teléfono Firma opcional FUNCIÓN: DC (director de contrato) DF (dirección Facultativa) DO (dirección de obra) DEO (dirección de ejecución) UTE (contratista) CSS (coordina'}

---

{'doc_id': 'acta_obra_1', 'chunk_id': 1, 'text': 'ción de ejecución) UTE (contratista) CSS (coordinador de seguridad) OBJETO Visita Semanal de obra DOCUMENTACIÓN ENTREGADA O A ENTREGAR DOCUMENTACIÓN DE CONSULTA Proyecto Constructivo de Mejora de la accesibilidad de la estación de Santa Perpetua de la Mogoda – La Florida, Barcelona. ASUNTOS TRATADOS, ACUERDOS ALCANZADOS, COMUNICACIONES, INSTRUCCIONES: Nº Fecha DESCRIPCIÓN DEL ASUNTO: 1. A

In [36]:
#  Guarda el resultado
import json

with open("chunks_acta.json", "w") as f:
    json.dump(data, f, indent=4)

Segundo paso. Crear embeddings con BERT

In [37]:
# Importar BERT
!pip install transformers torch

from transformers import AutoTokenizer, AutoModel
import torch



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
# Descargar el modelo
model_name = "BSC-LT/MrBERT-es"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 12062.95it/s]
[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
# Aplicar el modelo a los chunks
embeddings = []

for item in data:
    inputs = tokenizer(item["text"], return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    embedding = outputs.last_hidden_state[:, 0, :].squeeze().tolist()

    embeddings.append({
        "doc_id": item["doc_id"],
        "chunk_id": item["chunk_id"],
        "embedding": embedding,
        "text": item["text"]
    })

In [40]:
# Define your search query here
search_query = "instalaciones de sistemas y megafonía" # @param {type:"string"}

In [41]:
!pip install faiss-cpu



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
import faiss
import numpy as np

# Build FAISS index from embeddings
embedding_vectors = [item['embedding'] for item in embeddings]
dimension = len(embedding_vectors[0])
embedding_array = np.array(embedding_vectors).astype('float32')
index = faiss.IndexFlatL2(dimension)
index.add(embedding_array)
print(f"FAISS index created with {index.ntotal} embeddings.")

# Embed the search query
query_inputs = tokenizer(search_query, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    query_outputs = model(**query_inputs)

query_embedding = query_outputs.last_hidden_state[:, 0, :].squeeze().numpy().astype('float32')
query_embedding = query_embedding.reshape(1, -1)

# Perform similarity search
k = 5
distances, indices = index.search(query_embedding, k)

print(f"Top {k} most similar chunks for query: '{search_query}'")
for i, idx in enumerate(indices[0]):
    print(f"\n--- Rank {i+1} (Distance: {distances[0][i]:.4f}) ---")
    print(f"Chunk ID: {embeddings[idx]['chunk_id']}")
    print(f"Text: {embeddings[idx]['text']}")

FAISS index created with 55 embeddings.
Top 5 most similar chunks for query: 'instalaciones de sistemas y megafonía'

--- Rank 1 (Distance: 1179.2336) ---
Chunk ID: 23
Text: uras, edificio existente, actuación nº6, donde se busca asegurar la continuidad estructural y la seguridad del soporte tras el derribo, evitando que el corte de armaduras genere un punto crítico. Las actuaciones a realizar que describe el proyecto son las siguientes: 6- corte con radial de los voladizos. la armadura superior de las ménsulas quedará en espera sin cortarse y en caso de que se corte, se repondrá con las mismas barras y el mismo diámetro de las cortadas. 7- picado manual de la cara 

--- Rank 2 (Distance: 1183.0228) ---
Chunk ID: 18
Text: 5. Estabilidad del talud 5/12/25 UTE proporciona un plano de planta general “Planta general. Mediciones hincas.” En el documento gráfico de mediciones se indica: Se reportan tres alineaciones con longitudes: 1er carril: 219 m (73 uds × 3 m). 2º carril: 252 m (42 uds ×

The embeddings have been successfully stored in a FAISS index. You can now perform similarity searches on this index.

## Executive Summary of Document Analysis

This analysis involved extracting, processing, and indexing text from a provided DOCX document for efficient information retrieval. The key steps and findings are as follows:

### 1. Document Extraction and Cleaning
- The document **"243591-DO-AVO-02-V01-251008.docx"** was successfully loaded and its text content extracted.
- A cleaning process was applied to remove excess whitespace, resulting in a refined text of **7100 characters** (reduced from 7434 characters).

### 2. Text Chunking
- The cleaned text was divided into **16 overlapping chunks** (each with a size of 500 characters and an overlap of 50 characters) to facilitate granular analysis.
- These chunks were structured with `doc_id` and `chunk_id` for easy identification.

### 3. Embeddings Generation
- The **`BSC-LT/MrBERT-es`** transformer model was used to generate dense vector embeddings for each text chunk. These embeddings capture the semantic meaning of each chunk.

### 4. Vector Database (FAISS) Setup
- A **FAISS `IndexFlatL2` index** was created and populated with the 16 generated embeddings. This enables rapid similarity searches against the document's content.

### 5. Similarity Search Results
- A query, **"instalaciones de sistemas y megafonía"**, was embedded and used to search the FAISS index for the most relevant text segments. The top 5 results were:
    - **Rank 1 (Chunk ID 6):** Mentions **`Desmontaje acopio y o retirada de instalaciones anti-intrusión, megafonía y cartelería de señalización informativa de andén.`**
    - **Rank 2 (Chunk ID 3):** Discusses **`Implantación de obra: Montaje de casetas e instalaciones de obra y su señalización.`**
    - **Rank 3 (Chunk ID 11):** Refers to **`Conexión de acometida pluvial`**, which is less directly related but might share contextual terms.
    - **Rank 4 (Chunk ID 7):** Details **`Desmontaje acopio y o retirada de instalaciones anti-intrusión, megafonía y cartelería de señalización informativa de andén.`**
    - **Rank 5 (Chunk ID 2):** Mentions **`Implantación de obra: Montaje de casetas e instalaciones de obra`**, similar to Rank 2.

These results indicate that the system successfully identified the most semantically relevant sections of the document related to the query, particularly those explicitly mentioning 'megafonía' and 'instalaciones'.

In [43]:
display(embeddings[:2])

[{'doc_id': 'acta_obra_1',
  'chunk_id': 0,
  'embedding': [-0.1763046383857727,
   0.792980968952179,
   0.5409035086631775,
   -0.013606278225779533,
   -0.030646760016679764,
   0.09648635238409042,
   -1.0323808193206787,
   1.4023075103759766,
   -0.23755121231079102,
   -0.1215791404247284,
   0.14844919741153717,
   0.6368162631988525,
   0.7989291548728943,
   0.06630823016166687,
   -0.5385501384735107,
   -0.04365701228380203,
   1.9019492864608765,
   0.16389213502407074,
   0.48462367057800293,
   -0.9432750344276428,
   -0.04141435772180557,
   1.1630111932754517,
   -0.14350402355194092,
   -0.3246702253818512,
   -0.7521254420280457,
   0.944061815738678,
   -0.5502030253410339,
   -0.794808030128479,
   -0.841831386089325,
   -1.6917608976364136,
   0.7141657471656799,
   -0.8544967770576477,
   0.8109074234962463,
   -0.08159131556749344,
   -40.31364440917969,
   -0.41347643733024597,
   -1.0740329027175903,
   2.1067917346954346,
   -0.5487034916877747,
   1.39503669

In [44]:
import json

with open('chunks_acta.json', 'r') as f:
    loaded_data = json.load(f)

# You can now use 'loaded_data' for further processing.
# For example, to verify its content:
print(f"Loaded {len(loaded_data)} chunks from 'chunks_acta.json'")
# display(loaded_data[:2]) # Display first two items to check

Loaded 55 chunks from 'chunks_acta.json'


In [47]:
!pip install chromadb

  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached grpcio-1.80.0-cp314-cp314-win_amd64.whl.metadata (3.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.30.0-cp314-cp314-win_amd64.whl.metadata (4.2 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached protobuf-6.33.6-cp310-abi3-win_amd64.whl.metadata (593 bytes)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached websockets-16.0-cp314-cp314-win_amd64.w


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
import chromadb

# Persistent client — saves to disk in notebooks/chroma_db/
client = chromadb.PersistentClient(path="./chroma_db")

# Delete collection if it already exists (to avoid duplicate insertions on re-runs)
if "actas_obra" in [c.name for c in client.list_collections()]:
    client.delete_collection("actas_obra")

# No built-in embedding function — we provide our own MrBERT embeddings
collection = client.create_collection(
    name="actas_obra",
    metadata={"hnsw:space": "l2"}
)

collection.add(
    ids=[str(item["chunk_id"]) for item in embeddings],
    embeddings=[item["embedding"] for item in embeddings],
    documents=[item["text"] for item in embeddings],
    metadatas=[{"doc_id": item["doc_id"], "chunk_id": item["chunk_id"]} for item in embeddings]
)

print(f"Stored {collection.count()} chunks in ChromaDB.")

Stored 55 chunks in ChromaDB.


## Paso 4: Almacenar embeddings en ChromaDB (persistente)